In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from bayes_opt import BayesianOptimization
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')

def round_weights(weights):
    """Round weights according to specifications."""
    w1, w2, w3 = np.round(weights[:3], 2)
    wPL = np.round(weights[3], 7)
    return np.array([w1, w2, w3, wPL])

def prepare_data(data):
    """Prepare data for modeling: encode drive cycle and extract features."""
    le = LabelEncoder()
    data['DC_encoded'] = le.fit_transform(data['DC'])
    X = data[['w1', 'w2', 'w3', 'wPL', 'DC_encoded']]
    y = data['range']
    return X, y, le

def train_optimized_model(X, y):
    """
    Train an XGBoost model with hyperparameter tuning using a pipeline.
    Note: Early stopping parameters are removed from this step to avoid pipeline issues.
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
    ])
    
    param_space = {
        'xgb__max_depth': [3, 4, 5, 6, 7, 8],
        'xgb__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'xgb__n_estimators': [100, 200, 300, 400],
        'xgb__min_child_weight': [1, 3, 5],
        'xgb__subsample': [0.7, 0.8, 0.9, 1.0],
        'xgb__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'xgb__gamma': [0, 0.1, 0.2, 0.3],
        'xgb__reg_alpha': [0, 0.1, 0.5, 1],
        'xgb__reg_lambda': [1, 1.5, 2, 3]
    }
    
    random_search = RandomizedSearchCV(
        pipeline,
        param_space,
        n_iter=40,
        cv=5,
        scoring='neg_mean_squared_error',
        random_state=42,
        verbose=1,
        n_jobs=-1
    )
    
    random_search.fit(X_train, y_train)
    best_pipeline = random_search.best_estimator_
    print(f"Train R² Score: {best_pipeline.score(X_train, y_train):.4f}")
    print(f"Test R² Score: {best_pipeline.score(X_test, y_test):.4f}")
    return best_pipeline

def find_optimal_weights(model, dc_encoded, init_points=10, max_iter=100, tol=1e-3, patience=10):
    """
    Find optimal weights for a given drive cycle using Bayesian Optimization.
    This version uses a log-transformation for wPL and runs one iteration at a time,
    stopping if there is no significant improvement (tol) for 'patience' consecutive iterations.
    """
    def bo_objective(w1, w2, w3, log_wPL):
        # Convert log_wPL back to the original scale
        wPL = 10**log_wPL
        weights = np.array([
            np.round(w1, 2),
            np.round(w2, 2),
            np.round(w3, 2),
            np.round(wPL, 7)
        ])
        X = np.append(weights, dc_encoded).reshape(1, -1)
        return model.predict(X)[0]  # We maximize the predicted range

    # Define parameter bounds: w1, w2, w3 in [0,1] and log10(wPL) in [-7,-4]
    pbounds = {
        'w1': (0, 1),
        'w2': (0, 1),
        'w3': (0, 1),
        'log_wPL': (-7, -4)
    }
    
    optimizer = BayesianOptimization(
        f=bo_objective,
        pbounds=pbounds,
        random_state=42,
        verbose=2
    )
    
    # Run initial exploration
    optimizer.maximize(init_points=init_points, n_iter=0)
    
    no_improve_count = 0
    current_iter = 0
    prev_best = optimizer.max['target']
    
    # Continue running one iteration at a time until stopping criteria is met
    while current_iter < max_iter and no_improve_count < patience:
        optimizer.maximize(init_points=0, n_iter=1)
        current_best = optimizer.max['target']
        if current_best - prev_best < tol:
            no_improve_count += 1
        else:
            no_improve_count = 0
        prev_best = current_best
        current_iter += 1
    
    best_params = optimizer.max['params']
    best_wPL = 10**best_params['log_wPL']
    best_weights = np.array([
        np.round(best_params['w1'], 2),
        np.round(best_params['w2'], 2),
        np.round(best_params['w3'], 2),
        np.round(best_wPL, 7)
    ])
    best_range = optimizer.max['target']
    return best_weights, best_range

def optimize_all_dcs(data, model, le):
    """Find optimal weights for each drive cycle using Bayesian Optimization."""
    results = []
    for dc in data['DC'].unique():
        dc_encoded = le.transform([dc])[0]
        optimal_weights, max_range = find_optimal_weights(model, dc_encoded)
        results.append({
            'DC': dc,
            'w1': optimal_weights[0],
            'w2': optimal_weights[1],
            'w3': optimal_weights[2],
            'wPL': optimal_weights[3],
            'predicted_range': max_range
        })
    return pd.DataFrame(results)

def find_optimal_weights_for_dcs(data):
    """Main function: prepare data, train model, and optimize weights for each drive cycle."""
    X, y, le = prepare_data(data)
    print("Training model...")
    model = train_optimized_model(X, y)
    print("\nFinding optimal weights for each drive cycle using Bayesian Optimization...")
    results = optimize_all_dcs(data, model, le)
    return results, model, le

def predict_range(model, w1, w2, w3, wPL, dc, le):
    """Predict range for given weights and drive cycle using the trained pipeline."""
    w1, w2, w3 = np.round([w1, w2, w3], 2)
    wPL = np.round(wPL, 7)
    dc_encoded = le.transform([dc])[0]
    X = np.array([[w1, w2, w3, wPL, dc_encoded]])
    return model.predict(X)[0]


In [2]:
# === Main Execution ===
if __name__ == '__main__':
    # Read your dataset (ensure the CSV path is correct)
    data = pd.read_csv('./Version_2_Combined_DataSet.csv')
    
    # Train the model and find optimal weights for each drive cycle.
    results, trained_model, le = find_optimal_weights_for_dcs(data)
    print("\nOptimal weights for each drive cycle:")
    print(results)
    
    # Save results to a CSV file.
    results.to_csv('XGBOOST_Optimal_Weights_Bayesian_Enhanced.csv', index=False)
    
    # Example: Predict range for a sample configuration.
    sample_range = predict_range(trained_model, 0.8, 0.6, 0.4, 0.00005, 'NEDC_extended', le)
    print(f"\nPredicted range for sample configuration: {sample_range:.4f}")

Training model...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Train R² Score: 0.9999
Test R² Score: 0.9992

Finding optimal weights for each drive cycle using Bayesian Optimization...
|   iter    |  target   |  log_wPL  |    w1     |    w2     |    w3     |
-------------------------------------------------------------------------
| 1         | 221.4     | -5.876    | 0.9507    | 0.732     | 0.5987    |
| 2         | 217.5     | -6.532    | 0.156     | 0.05808   | 0.8662    |
| 3         | 220.6     | -5.197    | 0.7081    | 0.02058   | 0.9699    |
| 4         | 216.8     | -4.503    | 0.2123    | 0.1818    | 0.1834    |
| 5         | 219.9     | -6.087    | 0.5248    | 0.4319    | 0.2912    |
| 6         | 216.9     | -5.164    | 0.1395    | 0.2921    | 0.3664    |
| 7         | 221.0     | -5.632    | 0.7852    | 0.1997    | 0.5142    |
| 8         | 217.7     | -5.223    | 0.04645   | 0.6075    | 0.1705    |
| 9         | 221.5     | -6.805    | 0.9489    | 0.9656  

In [3]:
drive_cycles = ['FHDS_extended', 'FUDS_extended', 'FUDSfast_extended', 'NEDC_extended', 'US06City_extended', 'US06HW_extended', 'US06_extended']

results = results.sort_values(by='DC', key=lambda x: x.map({dc: i for i, dc in enumerate(drive_cycles)}))


In [4]:
results

,DC,w1,w2,w3,wPL,predicted_range
0,FHDS_extended,0.98,0.48,0.96,1.340000e-05,221.725403
2,FUDS_extended,0.16,0.06,0.87,3.000000e-07,167.856750
5,FUDSfast_extended,1.00,0.96,0.69,3.800000e-05,146.168030
4,NEDC_extended,0.16,0.06,0.87,3.000000e-07,168.916962
1,US06City_extended,0.98,0.48,0.96,1.340000e-05,171.004517
6,US06HW_extended,0.99,0.06,0.97,8.210000e-05,172.434433
3,US06_extended,0.99,0.40,0.99,8.700000e-05,175.677444


In [5]:
# sorting columns 
# w1,w2,w3,wPL,DC,predicted_range
results = results[['w1', 'w2', 'w3', 'wPL', 'DC', 'predicted_range']]

# saving wPL as string, so does not show as e- notation in csv

results['wPL'] = results['wPL'].apply(lambda x: '%.10f' % float(x))
results.to_csv('XGBOOST_MODEL_3_Optimal_Weights_Bayesian.csv', index=False)